In [1]:
import pandas as pd
import numpy as np

# Cargamos el archivo crudo (solo lectura, nunca lo modificamos)
df_crudo = pd.read_csv("../datos/crudos/application_train.csv")

# Copia de trabajo: aquí haremos todos los cambios
df = df_crudo.copy()

print(df.shape)

(307511, 122)


In [2]:
# 1) Bandera: 1 si tenía el código centinela, 0 si no
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)

# 2) Reemplazamos el código por NaN
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# 3) Verificación
print(df["DAYS_EMPLOYED_ANOM"].sum())
print(df["DAYS_EMPLOYED"].isnull().sum())
df["DAYS_EMPLOYED"].describe()

55374
55374


count    252137.000000
mean      -2384.169325
std        2338.360162
min      -17912.000000
25%       -3175.000000
50%       -1648.000000
75%        -767.000000
max           0.000000
Name: DAYS_EMPLOYED, dtype: float64

In [3]:
# Edad en años: cambiamos el signo y dividimos entre 365
df["EDAD_ANIOS"] = -df["DAYS_BIRTH"] / 365

# Antigüedad laboral en años (los NaN se mantienen como NaN)
df["ANTIGUEDAD_LABORAL_ANIOS"] = -df["DAYS_EMPLOYED"] / 365

# Verificación
df[["EDAD_ANIOS", "ANTIGUEDAD_LABORAL_ANIOS"]].describe().round(1)

,EDAD_ANIOS,ANTIGUEDAD_LABORAL_ANIOS
count,307511.0,252137.0
mean,43.9,6.5
std,12.0,6.4
min,20.5,-0.0
25%,34.0,2.1
50%,43.2,4.5
75%,53.9,8.7
max,69.1,49.1


In [4]:
# 1) Cuánto representa el crédito respecto al ingreso anual
df["RATIO_CREDITO_INGRESO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]

# 2) Cuánto representa la cuota anual respecto al ingreso anual
df["RATIO_CUOTA_INGRESO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]

# 3) Plazo implícito del crédito, en años: cuántas cuotas anuales hacen falta para pagarlo
df["PLAZO_ANIOS"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"]

# Verificación
df[["RATIO_CREDITO_INGRESO", "RATIO_CUOTA_INGRESO", "PLAZO_ANIOS"]].describe().round(2)

,RATIO_CREDITO_INGRESO,RATIO_CUOTA_INGRESO,PLAZO_ANIOS
count,307511.00,307499.00,307499.00
mean,3.96,0.18,21.61
std,2.69,0.09,7.82
min,0.00,0.00,8.04
25%,2.02,0.11,15.61
50%,3.27,0.16,20.00
75%,5.16,0.23,27.10
max,84.74,1.88,45.31


In [5]:
# Los 5 clientes con mayor ratio crédito/ingreso
cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
        "RATIO_CREDITO_INGRESO", "RATIO_CUOTA_INGRESO", "TARGET"]
display(df.nlargest(5, "RATIO_CREDITO_INGRESO")[cols])

# Cuántos clientes tienen un ratio mayor a 20 (unas 6 veces la mediana)
print((df["RATIO_CREDITO_INGRESO"] > 20).sum())

# El ingreso más bajo registrado
print(df["AMT_INCOME_TOTAL"].min())

,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,RATIO_CREDITO_INGRESO,RATIO_CUOTA_INGRESO,TARGET
20727,25650.0,2173500.0,48118.5,84.736842,1.875965,1
35791,45000.0,2215224.0,58567.5,49.227200,1.301500,0
226137,45000.0,1800000.0,62698.5,40.000000,1.393300,1
255247,58500.0,2146500.0,51340.5,36.692308,0.877615,1
158077,40500.0,1436850.0,42142.5,35.477778,1.040556,0


235
25650.0


In [6]:
print(df["RATIO_CREDITO_INGRESO"].mean())
print(df["RATIO_CREDITO_INGRESO"].median())
print(df.loc[df["RATIO_CREDITO_INGRESO"] <= 20, "RATIO_CREDITO_INGRESO"].mean())

3.957570238062716
3.265066666666667
3.941877039187382


In [7]:
bins_edad = [20, 30, 40, 50, 60, 70]
labels_edad = ["20-29", "30-39", "40-49", "50-59", "60-69"]

df["RANGO_EDAD"] = pd.cut(df["EDAD_ANIOS"], bins=bins_edad, labels=labels_edad, right=False)

df["RANGO_EDAD"].value_counts().sort_index()

RANGO_EDAD
20-29    45000
30-39    82299
40-49    76543
50-59    68074
60-69    35595
Name: count, dtype: int64

In [8]:
df["RANGO_INGRESO"] = pd.qcut(
    df["AMT_INCOME_TOTAL"],
    q=5,
    labels=["1-Muy bajo", "2-Bajo", "3-Medio", "4-Alto", "5-Muy alto"]
)

df["RANGO_INGRESO"].value_counts().sort_index()

# Los límites reales de cada tramo
pd.qcut(df["AMT_INCOME_TOTAL"], q=5).value_counts().sort_index()

AMT_INCOME_TOTAL
(25649.999, 99000.0]       63671
(99000.0, 135000.0]        85756
(135000.0, 162000.0]       35453
(162000.0, 225000.0]       75513
(225000.0, 117000000.0]    47118
Name: count, dtype: int64

In [9]:
df["AMT_INCOME_TOTAL"].value_counts().head(5)

AMT_INCOME_TOTAL
135000.0    35750
112500.0    31019
157500.0    26556
180000.0    24719
90000.0     22483
Name: count, dtype: int64

In [10]:
# 1) Categorías con valor "XNA" (código de "no informado" en Home Credit)
for col in df.select_dtypes(include=["object", "string"]).columns:
    n = (df[col] == "XNA").sum()
    if n > 0:
        print(f"{col}: {n} filas con XNA")

# 2) Valores imposibles en variables de conteo
print(df["CNT_CHILDREN"].max())
print(df["CNT_FAM_MEMBERS"].max())

# 3) Variables sobre el círculo social: ¿hay valores extremos?
df[["OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE"]].describe().round(1)

CODE_GENDER: 4 filas con XNA
ORGANIZATION_TYPE: 55374 filas con XNA
19
20.0


,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE
count,306490.0,306490.0
mean,1.4,0.1
std,2.4,0.4
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,2.0,0.0
max,348.0,34.0


In [11]:
# Banderas de "dato faltante" para las variables externas
for col in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    df[f"{col}_FALTANTE"] = df[col].isnull().astype(int)

# Revisión: tasa de incumplimiento según si falta o no EXT_SOURCE_1
df.groupby("EXT_SOURCE_1_FALTANTE")["TARGET"].mean().round(4)

EXT_SOURCE_1_FALTANTE
0    0.0750
1    0.0852
Name: TARGET, dtype: float64

In [12]:
print((df["CNT_CHILDREN"] >= 8).sum())
print(df["CNT_CHILDREN"].value_counts().sort_index().tail(8))

14
CNT_CHILDREN
7     7
8     2
9     2
10    2
11    1
12    2
14    3
19    2
Name: count, dtype: int64


In [13]:
print((df["OBS_30_CNT_SOCIAL_CIRCLE"] > 20).sum())
print((df["DEF_30_CNT_SOCIAL_CIRCLE"] > 5).sum())

102
14


In [14]:
df["CODE_GENDER"] = df["CODE_GENDER"].replace("XNA", np.nan)
print(df["CODE_GENDER"].value_counts(dropna=False))

CODE_GENDER
F      202448
M      105059
NaN         4
Name: count, dtype: int64


In [15]:
df.to_csv("../datos/procesados/application_train_limpio.csv", index=False)

# Verificación: releemos el archivo guardado
df_check = pd.read_csv("../datos/procesados/application_train_limpio.csv")
print(df_check.shape)

(307511, 133)
